# Task
Based on the provided plan, perform a comprehensive analysis of the global health data.

First, unzip "/content/Global Health Statistics 1.zip" and load the CSV into a DataFrame. Clean the data by renaming columns to standard formats, converting numeric types, and filling missing values with the yearly median. Aggregate the data by Country and Year.

Next, calculate three sub-scores using min-max scaling per year: 'BurdenScore', 'CapacityGap', and 'AffordabilityGap'. Combine these into a final 'StrainIndex' using weights of 0.5 for Burden, 0.3 for Capacity, and 0.2 for Affordability.

Finally, visualize the top 10 most strained Country-Years using a bar plot for the StrainIndex and a stacked bar chart for the sub-score contributions (using only matplotlib). Export the final DataFrame to "part1_strain_index_country_year.csv". Ensure the solution uses no machine learning libraries and avoids seaborn.

## Load and Inspect Data

### Subtask:
Unzip the provided zip file and load the extracted CSV into a pandas DataFrame for inspection.


**Reasoning**:
I need to unzip the file, find the CSV, and load it into a dataframe to proceed with the analysis.



In [ ]:
import zipfile
import os
import pandas as pd

zip_path = '/content/Global Health Statistics 1.zip'
extract_path = '/content/'

# Unzip the file
if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"Files extracted to {extract_path}")
else:
    print(f"File not found: {zip_path}")

# List files to find the CSV name
print("Files in directory:", os.listdir(extract_path))

# Assuming the CSV is named 'Global Health Statistics.csv' based on the zip name, but let's find it dynamically or fallback
# Since I don't know the exact name inside, I will look for a likely candidate.
csv_file = None
for file in os.listdir(extract_path):
    if file.endswith('.csv') and 'Global Health Statistics' in file:
        csv_file = file
        break

if csv_file:
    df = pd.read_csv(os.path.join(extract_path, csv_file))
    print(f"Loaded {csv_file}")
    print("Columns:", df.columns)
    display(df.head())
else:
    print("Could not find the expected CSV file.")

In [ ]:
# 00 — Config
import pandas as pd
import numpy as np
import json
import joblib
import pycountry
import os
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score
import matplotlib.pyplot as plt

CONFIG = {
    "CSV_FILE": 'Global Health Statistics.csv',
    "COL_MAPPING": {
        # Input columns from raw CSV
        'healthcare_access_': 'access',
        'doctors_per_1000': 'doctors',
        'hospital_beds_per_1000': 'beds',
        'per_capita_income_usd': 'income',
        'average_treatment_cost_usd': 'cost',
        # Computed/Target columns
        'CostToIncome': 'cost_to_income',
        'StrainIndex': 'strain_index',
        'BurdenScore': 'burden',
        'CapacityGap': 'capacity_gap',
        'AffordabilityGap': 'affordability_gap',
        'country': 'country',
        'year': 'year'
    },
    "FEATURES": ['access', 'doctors', 'beds', 'income', 'cost_to_income'],
    "TARGET": 'strain_index',
    "BUDGET": 100_000_000,
    "LEVERS": {
        'Access': {'feature': 'access', 'delta': 0.5, 'base_cost': 1_000_000, 'limit': 100.0},
        'Doctors': {'feature': 'doctors', 'delta': 0.01, 'base_cost': 1_500_000, 'limit': None},
        'Beds': {'feature': 'beds', 'delta': 0.02, 'base_cost': 1_200_000, 'limit': None}
    },
    "RF_PARAMS": {
        'n_estimators': 400,
        'min_samples_leaf': 2,
        'random_state': 42
    },
    "FILES": {
        'STRAIN_2024_JSON': 'strain_2024.json',
        'STRAIN_2024_CSV': 'strain_2024.csv',
        'MODEL_PKL': 'strain_model_rf.pkl',
        'WHATIF': 'whatif_2024.json',
        'BEST_ACTION': 'best_action_2024.json',
        'ALLOCATIONS': 'allocations_2024_rf.json',
        'AFTER': 'strain_2024_rf_after.json'
    }
}

print("Config loaded.")

In [ ]:
# 01 — Helpers

def get_iso3(name):
    try:
        return pycountry.countries.lookup(name).alpha_3
    except:
        return None

def add_iso3_column(df):
    df['iso3'] = df['country'].apply(get_iso3)
    manual_fixes = {'UK': 'GBR', 'Russia': 'RUS', 'Turkey': 'TUR', 'USA': 'USA', 'South Korea': 'KOR'}
    df['iso3'] = df['iso3'].fillna(df['country'].map(manual_fixes))
    return df

def scale_cost(income, median_income, base_cost):
    factor = income / median_income if median_income > 0 else 1.0
    return base_cost * factor

def predict_strain(model, row, features):
    # Helper to predict for a single row (Series or dict)
    # Convert to DataFrame to satisfy sklearn input requirements
    X = pd.DataFrame([row[features]], columns=features)
    return model.predict(X)[0]

def compute_whatif(model, df, features, levers):
    results = []
    for _, row in df.iterrows():
        base_strain = predict_strain(model, row, features)
        record = {
            "country": row['country'],
            "iso3": row['iso3'],
            "pred_strain_base": base_strain
        }

        for name, spec in levers.items():
            feat = spec['feature']
            # Simulate
            sim_row = row.copy()
            if spec['limit'] is None or sim_row[feat] < spec['limit']:
                sim_row[feat] += spec['delta']
                if spec['limit']: sim_row[feat] = min(sim_row[feat], spec['limit'])

            new_strain = predict_strain(model, sim_row, features)
            delta = max(0, base_strain - new_strain)

            record[f"pred_strain_if_{name.lower()}"] = new_strain
            record[f"delta_{name.lower()}"] = delta

        results.append(record)
    return pd.DataFrame(results)

def compute_best_action(df, features, levers, median_income, model):
    best_actions = []
    for _, row in df.iterrows():
        base_strain = predict_strain(model, row, features)
        best = None
        max_roi = -1.0

        for name, spec in levers.items():
            cost = scale_cost(row['income'], median_income, spec['base_cost'])

            # Check Limit
            feat = spec['feature']
            if spec['limit'] and row[feat] >= spec['limit']:
                continue

            # Simulate
            sim_row = row.copy()
            sim_row[feat] += spec['delta']

            new_strain = predict_strain(model, sim_row, features)
            delta = max(0, base_strain - new_strain)

            if cost > 0:
                roi = delta / cost
            else:
                roi = 0

            if roi > max_roi:
                max_roi = roi
                best = {
                    "best_lever": name,
                    "roi": roi,
                    "cost": cost,
                    "delta_pred_strain": delta
                }

        if best:
            best_actions.append({
                "iso3": row['iso3'],
                "country": row['country'],
                **best
            })

    return pd.DataFrame(best_actions)

def greedy_allocate(model, df_start, features, levers, budget, median_income):
    df_sim = df_start.copy()
    allocations = []

    iteration = 0
    # Determine stop condition: cheapest possible action across all countries
    min_base_cost = min(l['base_cost'] for l in levers.values())
    min_income_factor = df_sim['income'].min() / median_income
    min_possible_cost = min_base_cost * min_income_factor

    print(f"Starting Allocator. Budget: ${budget:,.2f}")

    while budget >= min_possible_cost:
        iteration += 1
        best_action = None
        global_max_roi = -1.0

        # Iterate through all countries to find global best next move
        for idx, row in df_sim.iterrows():
            current_strain = predict_strain(model, row, features)

            for name, spec in levers.items():
                # Check Constraints
                if spec['limit'] and row[spec['feature']] >= spec['limit']:
                    continue

                # Check Budget
                cost = scale_cost(row['income'], median_income, spec['base_cost'])
                if cost > budget:
                    continue

                # Simulate Prediction
                sim_row = row.copy()
                sim_row[spec['feature']] += spec['delta']
                new_strain = predict_strain(model, sim_row, features)

                delta = max(0, current_strain - new_strain)
                if delta == 0: continue

                roi = delta / cost

                if roi > global_max_roi:
                    global_max_roi = roi
                    best_action = {
                        "idx": idx,
                        "iso3": row['iso3'],
                        "country": row['country'],
                        "lever": name,
                        "cost": cost,
                        "pred_strain_before": current_strain,
                        "pred_strain_after": new_strain,
                        "delta": delta,
                        "feature": spec['feature'],
                        "delta_val": spec['delta']
                    }

        if not best_action:
            print("No beneficial actions remaining within budget.")
            break

        # Apply Best Action
        idx = best_action['idx']
        df_sim.loc[idx, best_action['feature']] += best_action['delta_val']
        budget -= best_action['cost']

        # Record
        allocations.append({
            "iteration": iteration,
            "iso3": best_action['iso3'],
            "country": best_action['country'],
            "lever": best_action['lever'],
            "cost": best_action['cost'],
            "pred_strain_before": best_action['pred_strain_before'],
            "pred_strain_after": best_action['pred_strain_after'],
            "delta_pred_strain": best_action['delta'],
            "remaining_budget": budget
        })

    return allocations, df_sim

print("Helpers loaded.")

In [223]:
# 02 — Part 1: Strain Index + 2024 Export

# A. Load and Clean
df_raw = pd.read_csv(CONFIG['CSV_FILE'])

# Standardize columns
df_raw.columns = df_raw.columns.str.lower().str.replace(' ', '_').str.replace(r'[^\w]', '', regex=True)

# Ensure numeric conversion & fill NaNs (Median per year)
numeric_cols = [
    'prevalence_rate_', 'incidence_rate_', 'mortality_rate_',
    'healthcare_access_', 'doctors_per_1000', 'hospital_beds_per_1000',
    'average_treatment_cost_usd', 'per_capita_income_usd', 'dalys'
]

for col in numeric_cols:
    if col in df_raw.columns:
        df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
        df_raw[col] = df_raw.groupby('year')[col].transform(lambda x: x.fillna(x.median()))

# B. Aggregate by Country-Year
agg_mean_cols = ['prevalence_rate_', 'incidence_rate_', 'mortality_rate_', 'dalys']
agg_median_cols = ['healthcare_access_', 'doctors_per_1000', 'hospital_beds_per_1000', 'per_capita_income_usd', 'average_treatment_cost_usd']

agg_mean = df_raw.groupby(['country', 'year'])[agg_mean_cols].mean().reset_index()
agg_median = df_raw.groupby(['country', 'year'])[agg_median_cols].median().reset_index()
agg_df = pd.merge(agg_mean, agg_median, on=['country', 'year'])

# C. Compute Features & Strain Index
agg_df['CostToIncome'] = agg_df['average_treatment_cost_usd'] / agg_df['per_capita_income_usd']

# Helper for Min-Max Scaling per year
def scale_col(data, col, invert=False):
    s = data.groupby('year')[col].transform(lambda x: (x - x.min()) / (x.max() - x.min()) if (x.max() - x.min()) != 0 else 0)
    return 1 - s if invert else s

# Sub-scores
burden_cols = ['prevalence_rate_', 'incidence_rate_', 'mortality_rate_', 'dalys']
capacity_cols = ['healthcare_access_', 'doctors_per_1000', 'hospital_beds_per_1000']

agg_df['BurdenScore'] = pd.DataFrame({c: scale_col(agg_df, c) for c in burden_cols}).mean(axis=1)
agg_df['CapacityGap'] = pd.DataFrame({c: scale_col(agg_df, c, invert=True) for c in capacity_cols}).mean(axis=1)
agg_df['AffordabilityGap'] = scale_col(agg_df, 'CostToIncome')

agg_df['StrainIndex'] = (0.5 * agg_df['BurdenScore']) + (0.3 * agg_df['CapacityGap']) + (0.2 * agg_df['AffordabilityGap'])

# D. Apply Column Mapping & ISO3
agg_df = agg_df.rename(columns=CONFIG['COL_MAPPING'])
agg_df = add_iso3_column(agg_df)

# E. Export 2024 Data
df_2024 = agg_df[agg_df['year'] == 2024].copy().reset_index(drop=True)

# Select only relevant columns for export
export_cols = ['country', 'iso3', 'year', 'strain_index', 'burden', 'capacity_gap', 'affordability_gap'] + CONFIG['FEATURES']
df_export = df_2024[export_cols].copy()

df_export.to_csv(CONFIG['FILES']['STRAIN_2024_CSV'], index=False)
df_export.to_json(CONFIG['FILES']['STRAIN_2024_JSON'], orient='records', indent=2)

print(f"Part 1 Done. 2024 Data Shape: {df_2024.shape}")
display(df_2024[['country', 'strain_index']].head())

Part 1 Done. 2024 Data Shape: (20, 17)


,country,strain_index
0,Argentina,0.599453
1,Australia,0.553320
2,Brazil,0.521664
3,Canada,0.505764
4,China,0.613601


In [224]:
# 03 — Part 2: Train Model (RF) + Evaluate

# A. Split Data (Time-based)
train_df = agg_df[agg_df['year'] <= 2022].copy()
test_df = agg_df[agg_df['year'] >= 2023].copy()

X_train = train_df[CONFIG['FEATURES']]
y_train = train_df[CONFIG['TARGET']]
X_test = test_df[CONFIG['FEATURES']]
y_test = test_df[CONFIG['TARGET']]

# B. Train Random Forest
rf_model = RandomForestRegressor(**CONFIG['RF_PARAMS'])
rf_model.fit(X_train, y_train)

# C. Evaluate
y_pred = rf_model.predict(X_test)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Model Trained. Test MAE: {mae:.4f}, R²: {r2:.4f}")

# D. Export Model
joblib.dump(rf_model, CONFIG['FILES']['MODEL_PKL'])
print(f"Model saved to {CONFIG['FILES']['MODEL_PKL']}")

Model Trained. Test MAE: 0.0660, R²: 0.2752
Model saved to strain_model_rf.pkl


In [225]:
# 04 — Part 3: What-if + Best Action (2024)

# A. What-If Analysis
whatif_df = compute_whatif(rf_model, df_2024, CONFIG['FEATURES'], CONFIG['LEVERS'])
whatif_df.to_json(CONFIG['FILES']['WHATIF'], orient='records', indent=2)
print(f"What-If analysis exported to {CONFIG['FILES']['WHATIF']}")

# B. Best Action per Country
median_income = df_2024['income'].median()
best_action_df = compute_best_action(df_2024, CONFIG['FEATURES'], CONFIG['LEVERS'], median_income, rf_model)
best_action_df.to_json(CONFIG['FILES']['BEST_ACTION'], orient='records', indent=2)
print(f"Best Actions exported to {CONFIG['FILES']['BEST_ACTION']}")

display(best_action_df[['country', 'best_lever', 'roi']].head())

What-If analysis exported to whatif_2024.json
Best Actions exported to best_action_2024.json


,country,best_lever,roi
0,Argentina,Access,2.163333e-08
1,Australia,Doctors,3.092056e-08
2,Brazil,Access,4.523432e-08
3,Canada,Doctors,3.798465e-09
4,China,Access,2.764675e-08


In [226]:
# 05 — Part 4: Greedy Allocator ($100M) + After Export

median_income = df_2024['income'].median()

allocations, df_after = greedy_allocate(
    rf_model,
    df_2024,
    CONFIG['FEATURES'],
    CONFIG['LEVERS'],
    CONFIG['BUDGET'],
    median_income
)

# A. Export Allocations
with open(CONFIG['FILES']['ALLOCATIONS'], 'w') as f:
    json.dump(allocations, f, indent=2)

# B. Export Final State
# We need to construct the 'after' view with predictions
final_records = []
for _, row in df_after.iterrows():
    # Recalculate predictions to be sure
    orig_row = df_2024[df_2024['iso3'] == row['iso3']].iloc[0]
    pred_before = predict_strain(rf_model, orig_row, CONFIG['FEATURES'])
    pred_after = predict_strain(rf_model, row, CONFIG['FEATURES'])

    final_records.append({
        "iso3": row['iso3'],
        "country": row['country'],
        "pred_strain_before": pred_before,
        "pred_strain_after": pred_after,
        "delta_pred_strain": max(0, pred_before - pred_after),
        "access_final": row['access'],
        "doctors_final": row['doctors'],
        "beds_final": row['beds']
    })

with open(CONFIG['FILES']['AFTER'], 'w') as f:
    json.dump(final_records, f, indent=2)

# C. Summary
total_spent = CONFIG['BUDGET'] - allocations[-1]['remaining_budget'] if allocations else 0
total_reduction = sum(r['delta_pred_strain'] for r in final_records)

print("\n--- Allocator Summary ---")
print(f"Total Actions: {len(allocations)}")
print(f"Total Spent: ${total_spent:,.2f}")
print(f"Global Strain Reduction: {total_reduction:.4f}")

df_res = pd.DataFrame(final_records)
top_10 = df_res.sort_values('delta_pred_strain', ascending=False).head(10)
display(top_10[['country', 'pred_strain_before', 'pred_strain_after', 'delta_pred_strain']])

Starting Allocator. Budget: $100,000,000.00

--- Allocator Summary ---
Total Actions: 81
Total Spent: $99,689,692.34
Global Strain Reduction: 1.0238


,country,pred_strain_before,pred_strain_after,delta_pred_strain
15,South Africa,0.493914,0.390176,0.103738
17,Turkey,0.502647,0.408615,0.094032
0,Argentina,0.516705,0.427903,0.088802
16,South Korea,0.585998,0.498358,0.087640
2,Brazil,0.437569,0.350390,0.087179
19,USA,0.513249,0.433719,0.079529
1,Australia,0.575677,0.497152,0.078525
10,Japan,0.442746,0.377212,0.065534
9,Italy,0.639534,0.588162,0.051372
4,China,0.514188,0.465877,0.048311


In [227]:
# 06 — Final Verification

required_files = list(CONFIG['FILES'].values())
missing = [f for f in required_files if not os.path.exists(f)]

if not missing:
    print("DONE: All steps completed successfully.")
    print("Generated Files:")
    for f in required_files:
        print(f" - {f}")
else:
    print("WARNING: Missing files:")
    for f in missing:
        print(f" - {f}")

DONE: All steps completed successfully.
Generated Files:
 - strain_2024.json
 - strain_2024.csv
 - strain_model_rf.pkl
 - whatif_2024.json
 - best_action_2024.json
 - allocations_2024_rf.json
 - strain_2024_rf_after.json
